# 07 · Decorators

From a first wrapper through `functools.wraps`, a real memoizing decorator, decorators with arguments (three layers), and stacking. Built on the closures from notebook 02.

### 7.1 — A first decorator

A decorator is a function that takes a function and returns a new function wrapping it. The `@deco` 
syntax above a definition is sugar for `f = deco(f)`.

Write `announce` that wraps a function so the returned wrapper appends `"called"` to a shared log 
before calling the original and returning its result.

In [2]:
def announce(func):
    def wrapper(*args, **kwargs):
        LOG.append("called")
        return func(*args, **kwargs)
    return wrapper

LOG = []

@announce
def add(a, b):
    return a + b

# Tests
LOG.clear()
assert add(2, 3) == 5          # original behaviour preserved
assert LOG == ["called"]       # wrapper ran
add(1, 1)
assert LOG == ["called", "called"]
print("7.1 passed")

7.1 passed


### 7.2 — Preserving metadata with `functools.wraps`

A naive wrapper hides the original function's name and docstring (`wrapper.__name__` becomes 
`"wrapper"`). `@functools.wraps(func)` on the wrapper copies that metadata across. This is why 
your earlier decorators felt slightly "off" — `wraps` is the fix.

Write `logged` using `@wraps(func)` so the decorated function keeps its original `__name__`.

In [3]:
from functools import wraps

def logged(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@logged
def greet(name):
    """Say hello."""
    return f"Hi {name}"

# Tests
assert greet("Mara") == "Hi Mara"
assert greet.__name__ == "greet"          # preserved by wraps
assert greet.__doc__ == "Say hello."      # docstring preserved too
print("7.2 passed")

7.2 passed


### 7.3 — A decorator that does real work

Write `cache` — a decorator that memoizes a single-argument function: on a repeat call with the same 
argument, it returns the stored result instead of recomputing. Track computations in a shared list 
so the test can confirm the function body ran only once per distinct input.

In [4]:
def cache(func):
    stored = {}
    def wrapper(n):
        if n in stored:
            return stored[n]
        else:
            stored[n] = func(n)
            return stored[n]
    return wrapper

COMPUTED = []

@cache
def square(n):
    COMPUTED.append(n)
    return n * n

# Tests
COMPUTED.clear()
assert square(4) == 16
assert square(4) == 16          # cached
assert square(5) == 25
assert COMPUTED == [4, 5]       # body ran once per distinct input
print("7.3 passed")

7.3 passed


### 7.4 — Decorators with arguments (three layers)

A decorator that takes its own argument needs **three** nested functions: the outer takes the 
argument, the middle takes the function, the inner is the wrapper. `@repeat(3)` means 
`repeat(3)` returns a decorator, which then decorates the function.

Write `repeat(times)` so the decorated function runs `times` times and returns a list of results.

In [5]:
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for i in range(times)]
        return wrapper
    return decorator

@repeat(3)
def shout(word):
    return word.upper()

# Tests
assert shout("hi") == ["HI", "HI", "HI"]

@repeat(2)
def double(x):
    return x * 2
assert double(5) == [10, 10]
print("7.4 passed")

7.4 passed


### 7.5 — Stacking decorators

Decorators stack bottom-up: the one nearest the function wraps first. Confirm you can reason about 
order. Write two decorators, `add_one` and `times_two`, each transforming a function's numeric return 
value, and observe the composition.

`@times_two` on top of `@add_one` means: take the function's result, add one, then double.

In [6]:
def add_one(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) + 1
    return wrapper

def times_two(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return wrapper

@times_two
@add_one
def get_value():
    return 10

# Tests
# add_one wraps first: 10 -> 11, then times_two: 11 -> 22
assert get_value() == 22
print("7.5 passed")

7.5 passed
